# Exploratory Data Analysis

In [1]:
import os
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                   # non-interactive backend — safe for scripts
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict

In [2]:
 
os.chdir("../..")       # for eda.ipynb            

In [13]:
#Load config
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
 
RAW_DIR     = Path(cfg["data"]["raw_dir"])   # data/raw/images/
PLOTS_DIR   = Path(cfg["eda"]["plots_dir"]) / "plots"             # notebooks/eda/plots
SEED        = cfg["data"]["seed"]                        # 42
CLASS_MAP   = cfg["classes"]                             # {Plastic:0, Paper:1, ...}
NUM_CLASSES = cfg["data"]["num_classes"]                 # 6
 
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

In [4]:
# Keyword → main class mapping 
KEYWORD_TO_CLASS = {
    # Plastic (9 sub-categories)
    "plastic"      : "Plastic",
    "bag"          : "Plastic",
    "straw"        : "Plastic",
    "cup lid"      : "Plastic",
    "cutlery"      : "Plastic",
    "container"    : "Plastic",
    "styrofoam"    : "Plastic",
    # Paper (5 sub-categories)
    "newspaper"    : "Paper",
    "office_paper" : "Paper",
    "office paper" : "Paper",
    "magazine"     : "Paper",
    "cardboard"    : "Paper",
    "paper_cup"    : "Paper",
    # Glass (3 sub-categories)
    "glass"        : "Glass",
    # Metal (4 sub-categories)
    "aluminum"     : "Metal",
    "steel"        : "Metal",
    "aerosol"      : "Metal",
    "metal"        : "Metal",
    # Organic (5 sub-categories)
    "food waste"   : "Organic",
    "food_waste"   : "Organic",
    "eggshell"     : "Organic",
    "coffee"       : "Organic",
    "tea bag"      : "Organic",
    "fruit"        : "Organic",
    "vegetable"    : "Organic",
    # Textile (2 sub-categories)
    "clothing"     : "Textile",
    "shoe"         : "Textile",
}
 
def get_main_class(subfolder_name: str) -> str:
    name_lower = subfolder_name.lower()
    for keyword, main_class in KEYWORD_TO_CLASS.items():
        if keyword in name_lower:
            return main_class
    return "Unknown"

In [5]:
# Step 1: Walk raw directory 
print("Scanning:", RAW_DIR)
 
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"\nDirectory not found: {RAW_DIR}\n"
        "Unzip the Kaggle dataset so that data/raw/images/<sub-category>/ exists."
    )
 
records = []
for subfolder in sorted(RAW_DIR.iterdir()):
    if not subfolder.is_dir():
        continue
    main_class = get_main_class(subfolder.name)
    for split_type in ["default", "real_world"]:
        split_dir = subfolder / split_type
        if not split_dir.exists():
            continue
        for img_path in split_dir.glob("*.png"):
            records.append({
                "subfolder"  : subfolder.name,
                "main_class" : main_class,
                "split_type" : split_type,
                "filepath"   : str(img_path),
            })
 
df = pd.DataFrame(records)
print(f"Total images found: {len(df)}")
 
unknown = df[df["main_class"] == "Unknown"]
if len(unknown) > 0:
    print(f"\nWARNING — {len(unknown)} images not mapped to any class:")
    print(unknown["subfolder"].unique())
    print("Add the missing keyword to KEYWORD_TO_CLASS in eda.py")
else:
    print("All sub-categories mapped successfully to 6 main classes.")

Scanning: data\raw\images
Total images found: 15000
All sub-categories mapped successfully to 6 main classes.


In [6]:
#  Step 2: Console summary 
print("\n" + "=" * 52)
print("  CLASS DISTRIBUTION SUMMARY")
print("=" * 52)
class_counts = df["main_class"].value_counts().reindex(CLASS_MAP.keys(), fill_value=0)
for cls, count in class_counts.items():
    pct = 100 * count / len(df)
    bar = "#" * int(pct / 2)
    print(f"  {cls:<10}  {count:>5}  ({pct:4.1f}%)  {bar}")
print("=" * 52)
 
print("\nSub-category image counts:")
sub_counts = (
    df.groupby(["main_class", "subfolder"])
    .size()
    .reset_index(name="count")
    .sort_values(["main_class", "count"], ascending=[True, False])
)
print(sub_counts.to_string(index=False))


  CLASS DISTRIBUTION SUMMARY
  Plastic      6500  (43.3%)  #####################
  Paper        3000  (20.0%)  ##########
  Glass        1000  ( 6.7%)  ###
  Metal        2000  (13.3%)  ######
  Organic      1500  (10.0%)  #####
  Textile      1000  ( 6.7%)  ###

Sub-category image counts:
main_class                  subfolder  count
     Glass     glass_beverage_bottles    500
     Glass            glass_food_jars    500
     Metal               aerosol_cans    500
     Metal         aluminum_food_cans    500
     Metal         aluminum_soda_cans    500
     Metal            steel_food_cans    500
   Organic             coffee_grounds    500
   Organic                  eggshells    500
   Organic                 food_waste    500
     Paper            cardboard_boxes    500
     Paper        cardboard_packaging    500
     Paper                  magazines    500
     Paper                  newspaper    500
     Paper               office_paper    500
     Paper                 paper_

In [7]:
#  Shared colour palette 
CLASS_COLORS = {
    "Plastic" : "#378ADD",
    "Paper"   : "#EF9F27",
    "Glass"   : "#1D9E75",
    "Metal"   : "#888780",
    "Organic" : "#639922",
    "Textile" : "#D4537E",
}
 
def save_fig(fig: plt.Figure, filename: str) -> None:
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")

In [8]:
#  Plot 1: Overall 6-class bar chart 
print("\n[1/5] Plotting class distribution...")
fig, ax = plt.subplots(figsize=(9, 5))
classes = list(class_counts.index)
counts  = list(class_counts.values)
colors  = [CLASS_COLORS[c] for c in classes]
bars    = ax.bar(classes, counts, color=colors, edgecolor="white", linewidth=0.6)
for bar, n in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 40,
        str(n), ha="center", va="bottom", fontsize=10
    )
ax.set_title("Image count per main class", fontsize=13, fontweight="bold", pad=10)
ax.set_ylabel("Number of images")
ax.set_ylim(0, max(counts) * 1.2)
sns.despine(ax=ax)
fig.tight_layout()
save_fig(fig, "class_distribution.png")
 


[1/5] Plotting class distribution...
  Saved → notebooks\eda\plots\class_distribution.png


In [9]:
# Plot 2: Sub-category distribution (6 subplots, one per class) 
print("[2/5] Plotting sub-category distribution...")
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
 
for idx, main_class in enumerate(CLASS_MAP.keys()):
    ax     = axes[idx]
    sub_df = sub_counts[sub_counts["main_class"] == main_class].copy()
 
    # Shorten sub-category names: strip the main class word for readability
    sub_df["short"] = sub_df["subfolder"].str.replace(
        main_class, "", case=False
    ).str.strip().replace("", main_class)
 
    color = CLASS_COLORS[main_class]
    ax.barh(sub_df["short"], sub_df["count"], color=color, edgecolor="white", linewidth=0.5)
    ax.axvline(500, color="gray", linestyle="--", linewidth=0.8, alpha=0.5, label="500 target")
    ax.set_title(main_class, fontsize=11, fontweight="bold")
    ax.set_xlabel("Images")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
 
fig.suptitle("Sub-category image counts grouped by class", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "subcategory_distribution.png")
 

[2/5] Plotting sub-category distribution...
  Saved → notebooks\eda\plots\subcategory_distribution.png


In [10]:
#  Plot 3: Default vs real_world 
print("[3/5] Plotting default vs real_world...")
pivot = (
    df.groupby(["main_class", "split_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(CLASS_MAP.keys())
)
 
fig, ax = plt.subplots(figsize=(10, 5))
x  = np.arange(len(pivot))
w  = 0.35
ax.bar(x - w/2, pivot.get("default",    pd.Series([0]*len(pivot))),
       width=w, label="default (studio)",    color="#378ADD", alpha=0.85)
ax.bar(x + w/2, pivot.get("real_world", pd.Series([0]*len(pivot))),
       width=w, label="real_world (in-use)", color="#D85A30", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=12)
ax.set_title("Default vs real-world counts per class", fontsize=13, fontweight="bold", pad=10)
ax.set_ylabel("Number of images")
ax.legend()
sns.despine(ax=ax)
fig.tight_layout()
save_fig(fig, "default_vs_realworld.png")

[3/5] Plotting default vs real_world...
  Saved → notebooks\eda\plots\default_vs_realworld.png


In [11]:
#  Plot 4: Sample image grid (3 rows × 6 cols)
print("[4/5] Building sample image grid...")
N_SAMPLES = 3
fig = plt.figure(figsize=(14, 7))
fig.suptitle(
    "Sample images per class (default + real_world mixed)",
    fontsize=13, fontweight="bold"
)
gs = gridspec.GridSpec(N_SAMPLES, NUM_CLASSES, figure=fig, hspace=0.3, wspace=0.05)
 
for col, main_class in enumerate(CLASS_MAP.keys()):
    class_df = df[df["main_class"] == main_class]
    samples  = class_df.sample(min(N_SAMPLES, len(class_df)), random_state=SEED)
    for row, (_, record) in enumerate(samples.iterrows()):
        ax = fig.add_subplot(gs[row, col])
        try:
            img = Image.open(record["filepath"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "load\nerror", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8)
        ax.axis("off")
        if row == 0:
            ax.set_title(
                main_class, fontsize=9, fontweight="bold",
                color=CLASS_COLORS[main_class], pad=4
            )
 
save_fig(fig, "sample_grid.png")

[4/5] Building sample image grid...
  Saved → notebooks\eda\plots\sample_grid.png


In [12]:
# Plot 5: Pixel brightness and std per class 
print("[5/5] Computing pixel statistics (sampling 100 images per class)...")
brightness_per_class = defaultdict(list)
std_per_class        = defaultdict(list)
 
for main_class in CLASS_MAP.keys():
    class_df = df[df["main_class"] == main_class]
    sample   = class_df.sample(min(100, len(class_df)), random_state=SEED)
    for _, row in sample.iterrows():
        try:
            arr = np.array(
                Image.open(row["filepath"]).convert("RGB").resize((64, 64))
            ) / 255.0
            brightness_per_class[main_class].append(arr.mean())
            std_per_class[main_class].append(arr.std())
        except Exception:
            continue
 
mean_brightness = {c: float(np.mean(v)) for c, v in brightness_per_class.items()}
mean_std        = {c: float(np.mean(v)) for c, v in std_per_class.items()}
 
print("\n  Pixel stats summary:")
print(f"  {'Class':<10}  {'Mean brightness':>16}  {'Mean std':>10}")
for c in CLASS_MAP.keys():
    print(f"  {c:<10}  {mean_brightness[c]:>16.4f}  {mean_std[c]:>10.4f}")
 
classes_ordered = list(CLASS_MAP.keys())
colors_ordered  = [CLASS_COLORS[c] for c in classes_ordered]
 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
axes[0].bar(classes_ordered, [mean_brightness[c] for c in classes_ordered],
            color=colors_ordered, edgecolor="white")
axes[0].set_title("Mean pixel brightness per class", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Mean brightness  (0 = black, 1 = white)")
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(classes_ordered, rotation=12)
sns.despine(ax=axes[0])
 
axes[1].bar(classes_ordered, [mean_std[c] for c in classes_ordered],
            color=colors_ordered, edgecolor="white")
axes[1].set_title("Mean pixel std deviation per class", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Std deviation  (higher = more varied colours)")
axes[1].set_ylim(0, 0.5)
axes[1].set_xticklabels(classes_ordered, rotation=12)
sns.despine(ax=axes[1])
 
fig.suptitle(
    "Pixel statistics — sample of 100 images per class",
    fontsize=13, fontweight="bold"
)
fig.tight_layout()
save_fig(fig, "pixel_stats.png")
 

[5/5] Computing pixel statistics (sampling 100 images per class)...

  Pixel stats summary:
  Class        Mean brightness    Mean std
  Plastic               0.7158      0.2398
  Paper                 0.7360      0.2609
  Glass                 0.7981      0.1990
  Metal                 0.7301      0.2654
  Organic               0.6692      0.3050
  Textile               0.6866      0.3022


C:\Users\USER\AppData\Local\Temp\ipykernel_10592\4067608542.py:37: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[0].set_xticklabels(classes_ordered, rotation=12)
C:\Users\USER\AppData\Local\Temp\ipykernel_10592\4067608542.py:45: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[1].set_xticklabels(classes_ordered, rotation=12)


  Saved → notebooks\eda\plots\pixel_stats.png
